# 01. LLM API Integration & Structured Outputs

Integrating commercial and open foundation LLMs via HTTP REST APIs and SDK clients.

### Key Design Principles:
- **Client Session Management**: Connection pooling and exponential backoff retry strategies
- **Structured JSON Outputs**: Enforcing strict Pydantic schema adherence with JSON Schema mode
- **Token Rate Limiting & Streaming**: Handling TPM (Tokens Per Minute) and RPM limits with Server-Sent Events (SSE)

In [ ]:
import json

# Simulated standard OpenAI/Anthropic/Gemini chat payload
def create_chat_payload(messages, model="gpt-4o", temperature=0.2, response_format=None):
    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature
    }
    if response_format:
        payload["response_format"] = response_format
    return payload

messages = [
    {"role": "system", "content": "You are a financial entity extraction assistant. Always reply in valid JSON."},
    {"role": "user", "content": "Apple Inc reported quarterly revenue of $85.8 billion, up 5% year over year."}
]

schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "FinancialExtraction",
        "schema": {
            "type": "object",
            "properties": {
                "company": {"type": "string"},
                "revenue_billions": {"type": "number"},
                "growth_pct": {"type": "number"}
            },
            "required": ["company", "revenue_billions", "growth_pct"]
        }
    }
}

payload = create_chat_payload(messages, response_format=schema)
print("Configured API Payload:\n", json.dumps(payload, indent=2))

## 🎯 Exercises & Practice
1. Explain how exponential backoff with jitter prevents the thundering herd problem during API rate limit spikes.

In [ ]:
# Exercise Solution
# Answer: Rather than all failed workers retrying at identical fixed intervals,
# exponential backoff doubles the wait time (2^attempt) and adds uniform random jitter,
# distributing retry requests smoothly across time and clearing API concurrency locks.